<a href="https://colab.research.google.com/github/AlKhrisW/JTIntern/blob/meisy/pemodelan/1_ForwardChaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# =============================================================================
#  SISTEM REKOMENDASI TEMPAT MAGANG
#  Metode : Forward Chaining
#  Mata Kuliah : Sistem Berbasis Pengetahuan (SBP)
# =============================================================================

import pandas as pd
import numpy as np
import re
from tabulate import tabulate          # pip install tabulate

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 0.  KONSTANTA & KONFIGURASI
# ─────────────────────────────────────────────────────────────────────────────

DATASET_PATH   = "PBL_DATA_preprocessed.xlsx"
SHEET_MHS      = "MAHASISWA"
SHEET_PRS      = "PERUSAHAAN"

# Bobot komponen skor (total = 100)
W_SKILL        = 0.60   # 60 % – kecocokan skill teknis
W_MINAT        = 0.25   # 25 % – keselarasan minat bidang
W_IPK          = 0.15   # 15 % – nilai IPK relatif

# Ambang batas minimum agar sebuah perusahaan masuk rekomendasi
MIN_SKILL_OVERLAP   = 1     # minimal 1 skill yang cocok
MIN_MATCH_RATIO     = 0.10  # minimal 10% skill perusahaan terpenuhi
DEFAULT_MIN_IPK     = 3.0   # jika perusahaan tidak menetapkan IPK (nilai 0)

TOP_N           = 5         # jumlah rekomendasi teratas yang ditampilkan

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1.  KAMUS NORMALISASI SKILL  (alias → token_standar)
# ─────────────────────────────────────────────────────────────────────────────
#
#  Tujuan : menyamakan variasi penulisan dari mahasiswa vs perusahaan.
#  Setiap key  = token standar yang dipakai dalam proses matching.
#  Setiap value = semua variant string yang merujuk ke teknologi yang sama.

SKILL_ALIASES = {
    # Bahasa pemrograman
    "python"        : ["python", "phyton", "pyhton", "py"],
    "javascript"    : ["javascript", "js", "javascript (vanilla)", "javascripts"],
    "typescript"    : ["typescript", "ts"],
    "php"           : ["php"],
    "java"          : ["java"],
    "kotlin"        : ["kotlin"],
    "dart"          : ["dart"],
    "go"            : ["go", "golang", "go / golang"],
    "csharp"        : ["c#", "c sharp"],
    "cpp"           : ["c++", "c / c++", "c/c++", "arduino"],
    "rust"          : ["rust"],
    "ruby"          : ["ruby"],
    "lua"           : ["lua"],
    "solidity"      : ["solidity"],
    "vb"            : ["visual basic", "vb.net", "visual basic / vb.net"],
    # Web framework / library
    "laravel"       : ["laravel"],
    "codeigniter"   : ["codeigniter", "ci"],
    "django"        : ["django"],
    "flask"         : ["flask"],
    "fastapi"       : ["fastapi", "fast api"],
    "express"       : ["express", "express.js", "node.js / express", "expressjs"],
    "nestjs"        : ["nestjs", "nest.js", "nest js"],
    "spring"        : ["spring", "spring boot"],
    "react"         : ["react", "react.js", "reactjs", "react js"],
    "vue"           : ["vue", "vue.js", "vuejs"],
    "angular"       : ["angular"],
    "nextjs"        : ["next.js", "nextjs", "next js"],
    "nuxtjs"        : ["nuxt.js", "nuxt"],
    "astro"         : ["astro", "astro.js"],
    "alpinejs"      : ["alpine.js", "alpinejs"],
    "jquery"        : ["jquery"],
    "bootstrap"     : ["bootstrap"],
    "tailwind"      : ["tailwind", "tailwindcss", "tailwind css"],
    "inertia"       : ["inertia.js", "inertia"],
    # Mobile
    "flutter"       : ["flutter"],
    "react_native"  : ["react native"],
    "android_studio": ["android studio"],
    # Database
    "mysql"         : ["mysql"],
    "postgresql"    : ["postgresql", "postgres", "postgre"],
    "sqlite"        : ["sqlite"],
    "sqlserver"     : ["sql server", "mssql", "microsoft sql server"],
    "mongodb"       : ["mongodb", "mongo"],
    "redis"         : ["redis"],
    "firebase"      : ["firebase"],
    "supabase"      : ["supabase"],
    "firestore"     : ["firestore"],
    "postgis"       : ["postgis"],
    "sql"           : ["sql"],
    # DevOps / Cloud / Infra
    "docker"        : ["docker"],
    "kubernetes"    : ["kubernetes", "k8s"],
    "git"           : ["git", "git & github", "github", "gitlab", "bitbucket"],
    "cicd"          : ["ci/cd", "cicd", "ci cd"],
    "aws"           : ["aws", "amazon web services"],
    "gcp"           : ["gcp", "google cloud", "google cloud platform"],
    "azure"         : ["azure", "microsoft azure"],
    "linux"         : ["linux", "linux server", "ubuntu", "debian"],
    "nginx"         : ["nginx"],
    "grafana"       : ["grafana"],
    "prometheus"    : ["prometheus"],
    # AI / ML / Data Science
    "tensorflow"    : ["tensorflow"],
    "pytorch"       : ["pytorch"],
    "sklearn"       : ["scikit-learn", "sklearn", "scikitlearn", "scikit learn"],
    "keras"         : ["keras"],
    "xgboost"       : ["xgboost"],
    "pandas"        : ["pandas"],
    "numpy"         : ["numpy"],
    "matplotlib"    : ["matplotlib", "matplotlib / seaborn", "seaborn"],
    "opencv"        : ["opencv"],
    "yolo"          : ["yolo", "yolov5", "yolov8"],
    "langchain"     : ["langchain"],
    "streamlit"     : ["streamlit"],
    "huggingface"   : ["hugging face", "huggingface"],
    # BI / Analitik
    "powerbi"       : ["power bi", "powerbi"],
    "tableau"       : ["tableau"],
    "looker"        : ["looker", "looker studio"],
    "excel"         : ["excel", "microsoft excel"],
    "pentaho"       : ["pentaho"],
    # Jaringan / Keamanan
    "cisco"         : ["cisco", "cisco / mikrotik", "mikrotik"],
    "wireshark"     : ["wireshark"],
    "nmap"          : ["nmap"],
    "kali"          : ["kali linux", "parrot os", "kali linux / parrot os"],
    "wazuh"         : ["wazuh"],
    "splunk"        : ["splunk"],
    "elk"           : ["elasticsearch", "elk", "elasticsearch / elk"],
    # Tools & lain-lain
    "figma"         : ["figma"],
    "postman"       : ["postman"],
    "bash"          : ["bash", "bash / shell", "shell"],
    "rest_api"      : ["rest api", "restful api", "rest", "api"],
    "graphql"       : ["graphql"],
    "websocket"     : ["websocket"],
    "uiux"          : ["ui/ux", "ui/ux design", "ux", "ui"],
    "uml"           : ["uml", "uml modeling"],
    "drawio"        : ["draw.io", "lucidchart", "draw.io / lucidchart"],
    "notion"        : ["notion", "trello", "jira", "trello / jira / notion"],
    "n8n"           : ["n8n"],
    "jupyter"       : ["jupyter", "jupyter notebook", "google colab"],
    "wordpress"     : ["wordpress"],
    "blender"       : ["blender", "blender / 3d"],
    "unity"         : ["unity", "unity (game dev)"],
    "canva"         : ["canva"],
}

# Peta minat mahasiswa → kata kunci posisi/bidang perusahaan
MINAT_KEYWORD_MAP = {
    "Software Developer"     : ["developer", "programmer", "engineer", "fullstack",
                                "backend", "frontend", "web developer", "software"],
    "Artificial Intelligence": ["ai", "artificial intelligence", "machine learning",
                                "ml", "deep learning", "llm", "data scientist",
                                "ai engineer"],
    "Data Technology"        : ["data", "analyst", "etl", "bi", "analytics",
                                "data engineer", "data analyst"],
    "Network and Security"   : ["network", "security", "soc", "infra", "cyber",
                                "it governance", "compliance"],
    "Multimedia and game"    : ["game", "ui/ux", "design", "multimedia", "creative",
                                "content", "social media"],
    "Information System"     : ["system analyst", "system", "erp", "bpm",
                                "information system"],
    "Business analyst"       : ["business", "analyst", "consulting", "data analyst"],
}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.  UTILITAS PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

def parse_list(raw) -> list:
    """Ubah string 'A, B, C' menjadi ['a', 'b', 'c'] (lowercase, stripped)."""
    if pd.isna(raw) or str(raw).strip() == "":
        return []
    return [s.strip().lower() for s in re.split(r",|;", str(raw)) if s.strip()]


def canonicalize(raw_list: list) -> set:
    """
    Petakan setiap skill mentah ke token standar menggunakan SKILL_ALIASES.
    Skill yang tidak ada di kamus tetap disimpan apa adanya (lowercase).
    """
    result = set()
    for token in raw_list:
        t = token.lower().strip()
        matched = False
        for canon, variants in SKILL_ALIASES.items():
            if any(v in t or t in v for v in variants):
                result.add(canon)
                matched = True
                break
        if not matched:
            result.add(t)
    return result


def load_dataset():
    """Muat dan preprocessing awal kedua sheet dari file Excel."""
    df_mhs = pd.read_excel(DATASET_PATH, sheet_name=SHEET_MHS)
    df_prs = pd.read_excel(DATASET_PATH, sheet_name=SHEET_PRS)

    # --- Mahasiswa ---
    df_mhs["IPK"] = pd.to_numeric(df_mhs["IPK"], errors="coerce").fillna(0.0)
    df_mhs["_skill_set"]  = df_mhs["Skill"].apply(lambda x: canonicalize(parse_list(x)))
    df_mhs["_minat_list"] = df_mhs["Minat_Bidang"].apply(parse_list)

    # --- Perusahaan ---
    df_prs["Minimal_IPK"] = pd.to_numeric(df_prs["Minimal_IPK"], errors="coerce").fillna(0)
    df_prs["_min_ipk"] = df_prs["Minimal_IPK"].apply(
        lambda v: DEFAULT_MIN_IPK if v == 0 else float(v)
    )
    # Gabung dua kolom skill sebagai referensi matching
    df_prs["_raw_skills"] = (
        df_prs["Skill_Dibutuhkan"].fillna("") + ", " +
        df_prs["Teknologi_Digunakan"].fillna("")
    )
    df_prs["_skill_set"] = df_prs["_raw_skills"].apply(
        lambda x: canonicalize(parse_list(x))
    )
    df_prs["_posisi_lower"] = (
        df_prs["Posisi_Magang"].fillna("").str.lower() + " " +
        df_prs["Bidang_Industri"].fillna("").str.lower()
    )
    return df_mhs, df_prs

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.  FUNGSI PENCOCOKAN MINAT
# ─────────────────────────────────────────────────────────────────────────────

def hitung_skor_minat(minat_list: list, posisi_lower: str) -> float:
    """
    Hitung proporsi minat mahasiswa yang selaras dengan posisi perusahaan.
    Nilai antara 0.0 – 1.0.
    """
    if not minat_list:
        return 0.0
    cocok = 0
    for minat in minat_list:
        keywords = MINAT_KEYWORD_MAP.get(minat, [minat.lower()])
        if any(kw in posisi_lower for kw in keywords):
            cocok += 1
    return cocok / len(minat_list)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.  FORWARD CHAINING ENGINE
# ─────────────────────────────────────────────────────────────────────────────

class ForwardChainingEngine:
    """
    Mesin inferensi Forward Chaining untuk rekomendasi tempat magang.

    Alur kerja (siklus match-resolve-act):
    ─────────────────────────────────────────
    Working Memory  <- fakta mahasiswa (skill, minat, IPK)
         |
         v
    [R1] Cek syarat IPK            -> GAGAL: perusahaan dilewati
         | LULUS
         v
    [R2] Hitung irisan skill       -> GAGAL (< MIN_SKILL_OVERLAP): dilewati
         | LULUS
         v
    [R3] Cek rasio kecocokan       -> GAGAL (< MIN_MATCH_RATIO): dilewati
         | LULUS
         v
    [R4] Cek keselarasan minat     -> skor parsial (0-1)
         |
         v
    [R5] Hitung skor gabungan      -> masukkan ke daftar kandidat
         |
         v
    Urutkan & kembalikan Top-N
    """

    def __init__(self, df_perusahaan: pd.DataFrame):
        self.df_prs = df_perusahaan

    # ── Definisi Aturan ──────────────────────────────────────────────────────

    @staticmethod
    def r1_syarat_ipk(ipk_mhs: float, min_ipk: float) -> bool:
        """
        R1 – Verifikasi IPK Minimum
        IF ipk_mahasiswa >= ipk_minimum_perusahaan THEN lolos tahap pertama.
        """
        return ipk_mhs >= min_ipk

    @staticmethod
    def r2_irisan_skill(skill_mhs: set, skill_prs: set):
        """
        R2 – Hitung Irisan Skill
        IF |skill_mahasiswa & skill_perusahaan| >= MIN_SKILL_OVERLAP
        THEN ada kecocokan teknis yang relevan.
        Mengembalikan (set_skill_cocok, jumlah_cocok).
        """
        cocok = skill_mhs & skill_prs
        return cocok, len(cocok)

    @staticmethod
    def r3_rasio_kecocokan(jumlah_cocok: int, total_skill_prs: int):
        """
        R3 – Uji Rasio Kecocokan
        IF (jumlah_cocok / total_skill_prs) >= MIN_MATCH_RATIO
        THEN mahasiswa memenuhi threshold relevansi teknis.
        """
        if total_skill_prs == 0:
            return True, 0.0
        rasio = jumlah_cocok / total_skill_prs
        return rasio >= MIN_MATCH_RATIO, round(rasio, 4)

    @staticmethod
    def r4_keselarasan_minat(minat_list: list, posisi_lower: str) -> float:
        """
        R4 – Evaluasi Keselarasan Minat
        IF minat_mahasiswa & bidang_perusahaan != kosong
        THEN ada kesesuaian jalur karir (skor 0-1).
        """
        return hitung_skor_minat(minat_list, posisi_lower)

    @staticmethod
    def r5_hitung_skor(
        rasio_skill: float,
        skor_minat: float,
        ipk_mhs: float,
        min_ipk: float,
    ) -> float:
        """
        R5 – Komputasi Skor Rekomendasi Gabungan
        IF R1, R2, R3, R4 terpenuhi
        THEN skor = W_SKILL*rasio_skill + W_MINAT*skor_minat + W_IPK*skor_ipk
        Skor akhir dalam rentang 0 – 100.
        """
        batas_atas = 4.0
        selisih    = batas_atas - min_ipk
        skor_ipk   = min((ipk_mhs - min_ipk) / selisih, 1.0) if selisih > 0 else 1.0
        skor_ipk   = max(skor_ipk, 0.0)

        skor = (W_SKILL * rasio_skill +
                W_MINAT * skor_minat  +
                W_IPK   * skor_ipk)
        return round(skor * 100, 2)

    # ── Proses Inferensi Utama ───────────────────────────────────────────────

    def infer(self, mahasiswa: pd.Series, top_n: int = TOP_N) -> list:
        """
        Jalankan siklus Forward Chaining untuk satu mahasiswa.
        Kembalikan list rekomendasi diurutkan berdasarkan skor tertinggi.
        """
        # ── Inisialisasi Working Memory ──
        skill_mhs = mahasiswa["_skill_set"]
        minat_mhs = mahasiswa["_minat_list"]
        ipk_mhs   = float(mahasiswa["IPK"])

        kandidat  = []

        for _, prs in self.df_prs.iterrows():

            jejak = []   # log aturan yang aktif (untuk traceability)

            min_ipk   = prs["_min_ipk"]
            skill_prs = prs["_skill_set"]
            posisi_lc = prs["_posisi_lower"]

            # ════════════════════════════════════════════
            #  R1 – Syarat IPK
            # ════════════════════════════════════════════
            if not self.r1_syarat_ipk(ipk_mhs, min_ipk):
                continue
            jejak.append(f"R1-PASS: IPK {ipk_mhs:.2f} >= {min_ipk:.1f}")

            # ════════════════════════════════════════════
            #  R2 – Irisan Skill
            # ════════════════════════════════════════════
            skill_cocok, jml_cocok = self.r2_irisan_skill(skill_mhs, skill_prs)
            if jml_cocok < MIN_SKILL_OVERLAP:
                continue
            sample = ", ".join(sorted(skill_cocok)[:3])
            jejak.append(f"R2-PASS: skill_match={jml_cocok} [{sample}]")

            # ════════════════════════════════════════════
            #  R3 – Rasio Kecocokan
            # ════════════════════════════════════════════
            lolos_r3, rasio = self.r3_rasio_kecocokan(jml_cocok, len(skill_prs))
            if not lolos_r3:
                continue
            jejak.append(f"R3-PASS: rasio={rasio:.1%} >= {MIN_MATCH_RATIO:.0%}")

            # ════════════════════════════════════════════
            #  R4 – Keselarasan Minat
            # ════════════════════════════════════════════
            skor_minat = self.r4_keselarasan_minat(minat_mhs, posisi_lc)
            jejak.append(f"R4: skor_minat={skor_minat:.1%}")

            # ════════════════════════════════════════════
            #  R5 – Hitung Skor Gabungan  ->  Tambah ke WM
            # ════════════════════════════════════════════
            skor = self.r5_hitung_skor(rasio, skor_minat, ipk_mhs, min_ipk)
            jejak.append(f"R5-PASS: skor_akhir={skor:.2f}/100")

            skill_kurang = sorted(skill_prs - skill_mhs)

            kandidat.append({
                "ID"           : prs["ID_Perusahaan"],
                "Perusahaan"   : prs["Nama_Perusahaan"],
                "Posisi"       : prs["Posisi_Magang"],
                "Bidang"       : prs["Bidang_Industri"],
                "Status"       : prs["Status_Paid"],
                "Durasi_Bulan" : prs["Durasi_Bulan"],
                "Skor"         : skor,
                "Skill_Cocok"  : sorted(skill_cocok),
                "Skill_Kurang" : skill_kurang[:5],
                "Rasio_Skill"  : rasio,
                "Skor_Minat"   : round(skor_minat, 4),
                "Jejak_Aturan" : jejak,
            })

        kandidat.sort(key=lambda x: x["Skor"], reverse=True)
        return kandidat[:top_n]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.  EVALUASI SISTEM
# ─────────────────────────────────────────────────────────────────────────────

def evaluasi_sistem(df_mhs: pd.DataFrame, engine: ForwardChainingEngine) -> dict:
    """
    Hitung metrik evaluasi untuk seluruh dataset mahasiswa.

    Metrik yang dihitung:
    - Coverage         : % mahasiswa yang mendapat >= 1 rekomendasi
    - Avg_Rekomendasi  : rata-rata jumlah rekomendasi per mahasiswa
    - Avg_Skor_Tertinggi : rata-rata skor peringkat-1 per mahasiswa
    - Avg_Skor_Semua   : rata-rata skor seluruh rekomendasi
    - Precision_Tinggi : % rekomendasi teratas dengan skor >= 50
    """
    total       = len(df_mhs)
    n_covered   = 0
    all_top     = []
    all_scores  = []
    n_confident = 0

    for _, mhs in df_mhs.iterrows():
        recs = engine.infer(mhs, top_n=TOP_N)
        if recs:
            n_covered += 1
            top_skor   = recs[0]["Skor"]
            all_top.append(top_skor)
            all_scores.extend(r["Skor"] for r in recs)
            if top_skor >= 50:
                n_confident += 1

    coverage    = n_covered / total * 100
    avg_recs    = len(all_scores) / max(n_covered, 1)
    avg_top     = float(np.mean(all_top))   if all_top    else 0.0
    avg_all     = float(np.mean(all_scores)) if all_scores else 0.0
    prec_tinggi = n_confident / max(n_covered, 1) * 100

    return {
        "Total Mahasiswa"                : total,
        "Mahasiswa Mendapat Rekomendasi" : n_covered,
        "Coverage (%)"                   : round(coverage, 2),
        "Rata-rata Jumlah Rekomendasi"   : round(avg_recs, 2),
        "Rata-rata Skor Tertinggi"       : round(avg_top, 2),
        "Rata-rata Skor Keseluruhan"     : round(avg_all, 2),
        "Precision Tinggi (Skor>=50) %"  : round(prec_tinggi, 2),
    }

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.  FUNGSI TAMPILAN / LAPORAN
# ─────────────────────────────────────────────────────────────────────────────

def cetak_header(judul: str) -> None:
    baris = "=" * 72
    print(f"\n{baris}")
    print(f"  {judul}")
    print(baris)


def cetak_rekomendasi(mhs: pd.Series, recs: list) -> None:
    """Cetak hasil rekomendasi satu mahasiswa ke konsol."""
    cetak_header(
        f"REKOMENDASI  |  {mhs['Nama']}  "
        f"(NIM: {mhs['NIM']} | IPK: {mhs['IPK']:.2f})"
    )

    if not recs:
        print("  [!] Tidak ada perusahaan yang memenuhi semua kriteria aturan.")
        return

    minat_str = str(mhs["Minat_Bidang"])
    skill_str = str(mhs["Skill"])
    print(f"  Minat  : {minat_str}")
    print(f"  Skill  : {skill_str[:80]}{'...' if len(skill_str) > 80 else ''}")
    print()

    for rank, r in enumerate(recs, start=1):
        cocok_str  = ", ".join(r["Skill_Cocok"][:6])
        kurang_str = ", ".join(r["Skill_Kurang"][:4])
        jejak_str  = " -> ".join(r["Jejak_Aturan"][:2])

        print(f"  +-- #{rank}  {r['Perusahaan']}")
        print(f"  |   Posisi  : {r['Posisi'][:65]}")
        print(f"  |   Bidang  : {r['Bidang']:18s}  Status: {r['Status']:10s}  Durasi: {r['Durasi_Bulan']} bln")
        print(f"  |   Skor    : {r['Skor']:.2f}/100  (skill {r['Rasio_Skill']:.0%} | minat {r['Skor_Minat']:.0%})")
        print(f"  |   Cocok   : {cocok_str  or '-'}")
        print(f"  |   Perlu   : {kurang_str or '-'}")
        print(f"  |   Jejak   : {jejak_str}")
        print(f"  +{'-'*69}")
    print()


def simpan_laporan_csv(df_mhs: pd.DataFrame, engine: ForwardChainingEngine,
                       path: str = "hasil_forward_chaining.csv") -> pd.DataFrame:
    """Jalankan inferensi semua mahasiswa dan simpan hasil ke CSV."""
    rows = []
    for _, mhs in df_mhs.iterrows():
        recs = engine.infer(mhs, top_n=3)
        if recs:
            for rank, r in enumerate(recs, 1):
                rows.append({
                    "NIM"            : mhs["NIM"],
                    "Nama"           : mhs["Nama"],
                    "IPK"            : mhs["IPK"],
                    "Rank"           : rank,
                    "ID_Perusahaan"  : r["ID"],
                    "Nama_Perusahaan": r["Perusahaan"],
                    "Posisi_Magang"  : r["Posisi"],
                    "Skor"           : r["Skor"],
                    "Rasio_Skill"    : f"{r['Rasio_Skill']:.2%}",
                    "Skor_Minat"     : f"{r['Skor_Minat']:.2%}",
                    "Skill_Cocok"    : ", ".join(r["Skill_Cocok"]),
                    "Skill_Kurang"   : ", ".join(r["Skill_Kurang"]),
                    "Status"         : r["Status"],
                    "Durasi_Bulan"   : r["Durasi_Bulan"],
                })
        else:
            rows.append({
                "NIM"            : mhs["NIM"],
                "Nama"           : mhs["Nama"],
                "IPK"            : mhs["IPK"],
                "Rank"           : 0,
                "ID_Perusahaan"  : "-",
                "Nama_Perusahaan": "Tidak ada rekomendasi",
                "Posisi_Magang"  : "-",
                "Skor"           : 0,
                "Rasio_Skill"    : "0%",
                "Skor_Minat"     : "0%",
                "Skill_Cocok"    : "-",
                "Skill_Kurang"   : "-",
                "Status"         : "-",
                "Durasi_Bulan"   : "-",
            })

    df_out = pd.DataFrame(rows)
    df_out.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"  [INFO] Laporan CSV disimpan -> {path}")
    return df_out

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 7.  MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    cetak_header("SISTEM REKOMENDASI TEMPAT MAGANG  -  FORWARD CHAINING")

    # ── Load & Preprocessing ──────────────────────────────────────────────────
    print("  [1] Memuat dataset ...")
    df_mhs, df_prs = load_dataset()
    print(f"      * {len(df_mhs)} data mahasiswa dimuat")
    print(f"      * {len(df_prs)} data perusahaan dimuat")

    # ── Inisialisasi Mesin Inferensi ──────────────────────────────────────────
    engine = ForwardChainingEngine(df_prs)
    print("\n  [2] Mesin Forward Chaining siap (5 aturan aktif: R1-R5)")

    # ── Demo: 4 Mahasiswa Representatif ──────────────────────────────────────
    print("\n  [3] Contoh inferensi per mahasiswa:\n")
    demo_idx = [0, 10, 50, 100]
    for idx in demo_idx:
        mhs  = df_mhs.iloc[idx]
        recs = engine.infer(mhs, top_n=3)
        cetak_rekomendasi(mhs, recs)

    # ── Evaluasi Sistem ───────────────────────────────────────────────────────
    print("\n  [4] Menghitung metrik evaluasi sistem ...")
    metrik = evaluasi_sistem(df_mhs, engine)
    cetak_header("EVALUASI SISTEM - FORWARD CHAINING")
    rows_m = [[k, v] for k, v in metrik.items()]
    print(tabulate(rows_m, headers=["Metrik", "Nilai"],
                   tablefmt="rounded_outline", colalign=("left", "right")))

    # ── Laporan Batch Semua Mahasiswa ─────────────────────────────────────────
    print("\n  [5] Menghasilkan laporan batch seluruh mahasiswa ...")
    df_hasil = simpan_laporan_csv(df_mhs, engine)

    # Tampilkan 10 baris pertama
    cetak_header("RINGKASAN LAPORAN (10 baris pertama)")
    cols = ["NIM", "Nama", "IPK", "Rank", "Nama_Perusahaan", "Skor", "Status"]
    print(tabulate(
        df_hasil[cols].head(10),
        headers=cols, tablefmt="rounded_outline", showindex=False,
    ))

    cetak_header("SELESAI")
    print("  Semua proses Forward Chaining berhasil diselesaikan.\n")


if __name__ == "__main__":
    main()


  SISTEM REKOMENDASI TEMPAT MAGANG  -  FORWARD CHAINING
  [1] Memuat dataset ...
      * 305 data mahasiswa dimuat
      * 30 data perusahaan dimuat

  [2] Mesin Forward Chaining siap (5 aturan aktif: R1-R5)

  [3] Contoh inferensi per mahasiswa:


  REKOMENDASI  |  Dinda SIta  (NIM: 848822 | IPK: 3.61)
  Minat  : Software Developer, Artificial Intelligence, Data Technology, Multimedia and game, Information System, Business analyst
  Skill  : Python, JavaScript, TypeScript, Java, SQL, Tailwind CSS, Bootstrap, HTML & CSS, ...

  +-- #1  DOT Indonesia
  |   Posisi  : Backend Programmer / Frontend Programmer / Fullstack Programmer /
  |   Bidang  : swasta nasional     Status: Paid        Durasi: 5 bln
  |   Skor    : 43.43/100  (skill 57% | minat 0%)
  |   Cocok   : figma, javascript, mysql, python
  |   Perlu   : desain, react, rest_api
  |   Jejak   : R1-PASS: IPK 3.61 >= 3.0 -> R2-PASS: skill_match=4 [figma, javascript, mysql]
  +-------------------------------------------------------